# Bruker -> NIfTI conversion (slice-gap double-count fixed)

**The bug.** `brkraw` 0.5.7 counts the inter-slice gap twice on the through-plane
axis. In `brkraw/resolver/affine.py`:

| line | code | note |
|------|------|------|
| 476 | `slice_thickness = method.get('PVM_SPackArrSliceDistance')` | this is centre-to-centre distance, **already** thickness + gap |
| 478 | `slice_gap = method.get('PVM_SPackArrSliceGap')` | |
| 542 | `slice_thickness = [t + slice_gap[i] for i, t in enumerate(slice_thickness)]` | **adds the gap a second time** |

The inflated value becomes the z extent (line 413: `extent = num_slices * slice_thickness`)
and then the z voxel size (line 448: `resols = extent / shape`), so it corrupts
**the affine itself**, not merely the header `pixdim`.

The misleading part is the variable name: `slice_thickness` never holds
`PVM_SliceThick`. The comment on line 541 (*"slice thickness = image thickness +
slice gap"*) describes arithmetic that would be right if it did.

**Observed effect**

| scan | thickness | gap | true spacing | brkraw wrote |
|------|-----------|-----|--------------|--------------|
| NHP1 scan 9 | 0.60 mm | 0.15 mm | **0.75 mm** | 0.90 mm (x1.20) |
| NHP2 scan 3 | 0.75 mm | 0.00 mm | **0.75 mm** | 0.75 mm (correct) |

A zero gap hides the bug entirely, which is why only *some* scans were inflated.
3D acquisitions (`VisuCoreDim == 3`) are also unaffected -- they take a different
branch that reads `VisuCoreExtent` directly. So the scans at risk are exactly the
**2D multi-slice scans acquired with a non-zero gap**.

**The fix** is in the cell titled *THE FIX*: `resolve_slice_pack` is wrapped so the
gap it reports is 0, because the distance it also reports already contains it.
That single change makes the affine and the header correct at conversion time.

**Retired cells.** The old `_zfix` rescaler and its crop-alignment check are gone.
They rewrote finished files for one hand-picked scan; conversion is now correct at
the source, so re-run this notebook from the top and every scan comes out right.
Regenerate any downstream derivatives (skull-strips, crops, labels) from the new
outputs -- a mask drawn on a 0.90 mm volume is not valid on a 0.75 mm one.

> Run the cells in order. *THE FIX* must execute before any conversion.


In [ ]:
# 1. Mount Drive, unzip, install brkraw, locate the study folders
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os, glob

ZIP_PATH   = '/content/drive/My Drive/EKAM_NHP.zip'   # adjust if needed
EXTRACT_TO = '/content/EKAM'
os.makedirs(EXTRACT_TO, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_TO)

# pin the version this notebook was written against
!pip install -q brkraw==0.5.7

studies = sorted(glob.glob('/content/EKAM/**/*EKAM_NHP*', recursive=True))
studies = [s for s in studies
           if os.path.isdir(s) and os.path.exists(os.path.join(s, 'subject'))]
print("Found studies:")
for s in studies:
    print("  ", s)


## THE FIX

Wraps `brkraw.resolver.affine.resolve_slice_pack` so the reported gap is 0. Line
542 then adds nothing and `slice_thickness` keeps its true meaning: the
centre-to-centre slice distance.

Three safeguards:

* **Guarded** -- it inspects the installed source and only patches if *both* the
  gap re-addition and the `PVM_SPackArrSliceDistance` source are present, so a
  future brkraw that fixes this upstream will not be double-corrected.
* **Idempotent** -- re-running the cell will not wrap the wrapper.
* **Fallback** -- if `PVM_SPackArrSliceDistance` is missing or zero, spacing is
  rebuilt as `PVM_SliceThick + PVM_SPackArrSliceGap` instead of collapsing to 0.

It ends with a self-test on synthetic parameters (NHP1's 0.6 mm / 0.15 mm case)
that asserts 0.90 mm before and 0.75 mm after, then restores everything it
stubbed. No scan data required.

In [ ]:
# 2. THE FIX - patch the slice-gap double count. Run before any conversion.
import inspect
import numpy as np
import brkraw
import brkraw.resolver.affine as _affine

print("brkraw", brkraw.__version__)

# Stash the untouched original so this cell can be re-run safely. Everything
# below inspects _ORIG, never the live attribute, which may already be our
# exec-defined replacement (that has no retrievable source).
_ORIG = getattr(_affine, '_ORIG_resolve_slice_pack', _affine.resolve_slice_pack)
_affine._ORIG_resolve_slice_pack = _ORIG

def _src_of(fn):
    try:
        return inspect.getsource(fn)
    except (OSError, TypeError):
        return ''

# --- guard: only patch a brkraw that actually has the bug --------------------
_gap_readded = 't + slice_gap[i]' in _src_of(_affine.resolve)
_dist_source = 'PVM_SPackArrSliceDistance' in _src_of(_ORIG)
_buggy = _gap_readded and _dist_source

if not _buggy:
    print("!! This brkraw does not match the known-buggy pattern "
          f"(gap re-added={_gap_readded}, distance-sourced={_dist_source}).")
    print("   NOT patching - it may already be fixed upstream. The self-test "
          "below still checks the result.")
else:
    def _as_list(v):
        if v is None:
            return []
        if isinstance(v, np.ndarray):
            return v.tolist()
        if isinstance(v, (list, tuple)):
            return list(v)
        return [v]

    def resolve_slice_pack_fixed(scan):
        """PVM_SPackArrSliceDistance already includes the gap -> report gap = 0."""
        info = _ORIG(scan)
        if not info:
            return info

        dist = _as_list(info.get('slice_thickness'))   # = PVM_SPackArrSliceDistance
        gap  = _as_list(info.get('slice_gap'))
        npack = max(len(_as_list(info.get('num_slices'))), len(dist), 1)

        # only needed for the degenerate no-distance fallback
        try:
            thick = _as_list(_affine.get_file(scan, 'method').get('PVM_SliceThick'))
        except Exception:
            thick = []

        def _at(seq, i, default=0.0):
            if not seq:
                return default
            return float(seq[i] if i < len(seq) else seq[-1])

        spacing = []
        for i in range(npack):
            d, g = _at(dist, i), _at(gap, i)
            spacing.append(d if d > 0 else _at(thick, i) + g)

        info['slice_thickness'] = spacing      # true centre-to-centre distance
        info['slice_gap']       = [0.0] * npack  # neutralise the re-add on L542
        return info

    _affine.resolve_slice_pack = resolve_slice_pack_fixed
    print("patched brkraw.resolver.affine.resolve_slice_pack")


# --- self-test: NHP1-like geometry, no scan data needed ----------------------
class _P:
    def __init__(self, d): self.d = d
    def get(self, k):      return self.d.get(k)
    def search_keys(self, s): return ['PVM_SPackArrSliceDistance']

_N = 73
_method = _P({'PVM_NSPacks': 1, 'PVM_SPackArrNSlices': [_N],
              'PVM_SPackArrSliceDistance': [0.75],   # = 0.6 thick + 0.15 gap
              'PVM_SPackArrSliceGap': [0.15], 'PVM_SliceThick': [0.6],
              'PVM_SPackArrSliceOrient': 'axial'})
_acqp = _P({'ACQ_scaling_phase': 1})
_visu = _P({'VisuCoreDim': 2, 'VisuCoreSize': [512, 512],
            'VisuCoreExtent': [95.0, 95.0],
            'VisuCoreOrientation': np.tile(np.eye(3).ravel(), (_N, 1)),
            'VisuCorePosition': np.column_stack(
                [np.zeros(_N), np.zeros(_N), np.arange(_N) * 0.75]),
            'VisuCoreSlicePacksSlices': None,
            'VisuSubjectType': 'Monkey', 'VisuSubjectPosition': 'Head_Prone'})

def _measure(pack_fn):
    """Run resolve() with mocked parameter files and the given slice-pack fn."""
    _saved = (_affine.get_file, _affine.get_reco, _affine.resolve_slice_pack)
    try:
        _affine.get_reco = lambda scan, rid: object()
        _affine.get_file = lambda obj, name: {'acqp': _acqp, 'method': _method,
                                              'visu_pars': _visu}[name]
        _affine.resolve_slice_pack = pack_fn
        res = _affine.resolve(object(), 1)
        return float(np.linalg.norm(res['affines'][0][:3, 2]))
    finally:
        _affine.get_file, _affine.get_reco, _affine.resolve_slice_pack = _saved

_stock = _measure(_ORIG)                          # unpatched behaviour
_live  = _measure(_affine.resolve_slice_pack)     # what conversion will now use

print(f"\nself-test (0.6 mm slice, 0.15 mm gap -> true spacing 0.75 mm)")
print(f"   stock brkraw : {_stock:.4f} mm")
print(f"   in effect now: {_live:.4f} mm")

if _buggy:
    assert abs(_stock - 0.90) < 1e-6, f"expected stock 0.90, got {_stock}"
    assert abs(_live  - 0.75) < 1e-6, f"expected patched 0.75, got {_live}"
    print("   PASS - gap no longer double counted; affine z scale corrected "
          "(not just the header)")
else:
    assert abs(_live - 0.75) < 1e-6, (
        f"brkraw was not patched and still reports {_live} mm instead of 0.75 mm - "
        "inspect resolver/affine.py before converting anything")
    print("   PASS - this brkraw already reports the correct spacing; no patch needed")


## Bruker parameter helpers + diagnostic

Defines `parse_jcamp` / `val` / `first` / `expected_spacing`, used by the
conversion and verification cells below, so this cell is **required**, not
optional. `expected_spacing` returns ground truth per slice pack, preferring the
actual slice positions in `acqp` (`ACQ_slice_offset`) and falling back to
`PVM_SPackArrSliceDistance`.

In [ ]:
# 3. Bruker parameter helpers + pre-conversion diagnostic
import os, re, glob
import numpy as np, nibabel as nib

def parse_jcamp(path):
    """Parse a Bruker JCAMP-DX parameter file -> {name: (shape, raw_value_string)}."""
    if not os.path.exists(path):
        return None
    params, key, buf, shape = {}, None, [], None
    def flush():
        nonlocal key, buf, shape
        if key is not None:
            params[key] = (shape, " ".join(buf).strip())
        key, buf, shape = None, [], None
    with open(path, errors="ignore") as fh:
        for line in fh:
            line = line.rstrip("\n")
            m = re.match(r"##\$?([\w\[\]]+)=(.*)$", line)
            if m:
                flush()
                key, rest = m.group(1), m.group(2).strip()
                sm = re.match(r"\(\s*([\d,\s]+)\)\s*(.*)$", rest)
                if sm:
                    shape = tuple(int(x) for x in sm.group(1).replace(",", " ").split())
                    buf = [sm.group(2)]
                else:
                    shape, buf = None, [rest]
            elif line.startswith(("##", "$$")):
                flush()
            elif key is not None:
                buf.append(line)
    flush()
    return params

def val(p, name):
    if not p or name not in p:
        return None
    toks = p[name][1].split()
    if not toks:
        return None
    try:
        nums = [float(t) for t in toks]
        return nums[0] if len(nums) == 1 else nums
    except ValueError:
        return toks[0] if len(toks) == 1 else toks

def first(v):
    return v[0] if isinstance(v, list) and v else v

def as_list(v):
    if v is None: return []
    return list(v) if isinstance(v, list) else [v]

def fmt(v, unit=""):
    if v is None: return "n/a"
    if isinstance(v, list):
        head = v[:6]
        s = ", ".join(f"{x:g}" if isinstance(x, float) else str(x) for x in head)
        if len(v) > 6: s += f", ... ({len(v)} values)"
        return f"[{s}]{unit}"
    return (f"{v:g}" if isinstance(v, float) else str(v)) + unit


def study_prefix(path, idx):
    """NHP1 / NHP2 / ... derived from the study folder name."""
    m = re.search(r'(NHP\d+)', os.path.basename(path))
    return m.group(1) if m else f'study{idx + 1}'


def expected_spacing(study, scan, reco=1):
    """Ground-truth through-plane spacing per slice pack.

    Returns (list_of_spacings, source_string, is_2d). For 3D scans the value comes
    from VisuCoreExtent/VisuCoreSize and the slice-gap bug does not apply.
    """
    method = parse_jcamp(f"{study}/{scan}/method")
    acqp   = parse_jcamp(f"{study}/{scan}/acqp")
    visu   = parse_jcamp(f"{study}/{scan}/pdata/{reco}/visu_pars")
    if method is None:
        return [], "no method file", None

    dim = first(val(visu, 'VisuCoreDim')) if visu else None
    if dim == 3:
        ext  = as_list(val(visu, 'VisuCoreExtent'))
        size = as_list(val(visu, 'VisuCoreSize'))
        if len(ext) >= 3 and len(size) >= 3 and size[2]:
            return [ext[2] / size[2]], "VisuCoreExtent/VisuCoreSize (3D)", False
        return [], "3D, indeterminate", False

    dist   = as_list(val(method, 'PVM_SPackArrSliceDistance'))
    nsl    = [int(x) for x in as_list(val(method, 'PVM_SPackArrNSlices'))] or [1]
    offs   = as_list(val(acqp, 'ACQ_slice_offset')) if acqp else []

    # prefer measured slice positions, pack by pack
    if offs and len(offs) == sum(nsl):
        out, ok, i = [], True, 0
        for n in nsl:
            seg = np.asarray(offs[i:i + n], float); i += n
            if n > 1:
                d = np.diff(np.sort(seg))
                if np.ptp(d) < 1e-6:
                    out.append(float(d[0]))
                else:
                    ok = False; break
            else:
                out.append(None)          # single slice: no measurable step
        if ok:
            filled = [o if o is not None else (dist[k] if k < len(dist) else None)
                      for k, o in enumerate(out)]
            if all(f is not None for f in filled):
                n_meas = sum(o is not None for o in out)
                if n_meas == len(out):
                    src = "ACQ_slice_offset steps"
                elif n_meas == 0:
                    src = "PVM_SPackArrSliceDistance"
                else:
                    src = "ACQ_slice_offset steps + PVM_SPackArrSliceDistance"
                return [float(f) for f in filled], src, True

    if dist:
        return [float(d) for d in dist], "PVM_SPackArrSliceDistance", True
    return [], "indeterminate", True


def report(label, study, scan, reco=1):
    print("=" * 74)
    print(f"{label}  --  scan {scan}, reco {reco}")
    print(f"  {study}")
    print("=" * 74)
    method = parse_jcamp(f"{study}/{scan}/method")
    visu   = parse_jcamp(f"{study}/{scan}/pdata/{reco}/visu_pars")
    if method is None:
        print("  !! no method file at that path -- check the scan number\n"); return

    th = first(val(method, 'PVM_SliceThick'))
    gp = first(val(method, 'PVM_SPackArrSliceGap'))
    di = first(val(method, 'PVM_SPackArrSliceDistance'))
    print(f"  sequence : {fmt(val(method,'Method'))}")
    print(f"    PVM_SliceThick            = {fmt(th,' mm')}")
    print(f"    PVM_SPackArrSliceGap      = {fmt(gp,' mm')}")
    print(f"    PVM_SPackArrSliceDistance = {fmt(di,' mm')}")
    print(f"    PVM_SPackArrNSlices       = {fmt(val(method,'PVM_SPackArrNSlices'))}")
    if isinstance(th, float) and isinstance(gp, float):
        print(f"    -> thickness + gap = {th + gp:g} mm   (distance says {fmt(di,' mm')})")

    exp, src, is2d = expected_spacing(study, scan, reco)
    print(f"  ground truth spacing : {fmt(exp,' mm')}   [{src}]")
    if is2d and isinstance(gp, float) and gp > 0 and exp:
        print(f"  unpatched brkraw would have written {exp[0] + gp:g} mm "
              f"(x{(exp[0] + gp) / exp[0]:.3f}) -- this scan was affected")
    elif is2d:
        print("  gap is 0 -> this scan was NOT affected by the bug")
    else:
        print("  3D acquisition -> not affected by the bug")
    print()


for lbl, scan in [("NHP1", 9), ("NHP2", 3)]:
    hits = [p for p in glob.glob(f"/content/EKAM/**/*{lbl}*", recursive=True)
            if os.path.isdir(p) and os.path.exists(os.path.join(p, "subject"))]
    if not hits:
        print(f"!! could not locate the {lbl} study folder under /content/EKAM\n")
    else:
        report(lbl, sorted(hits)[0], scan)


In [ ]:
# 3b. Orientation audit -- what every scan claims about its own geometry
# =====================================================================================
# The slice-gap fix corrects the through-plane SCALE. This cell is about the through-plane
# DIRECTION, which is a separate failure and is not detectable from spacing.
#
# PVM_SPackArrSliceOrient / PVM_SPackArrReadOrient are the slice package's plane and read
# direction in the GRADIENT frame. VisuSubjectPosition + VisuSubjectType are what maps the
# gradient frame onto the animal. VisuCoreOrientation holds the per-slice direction cosines
# brkraw actually builds the affine from. If two scans of one animal disagree on any of
# these, the resulting NIfTI files sit in different anatomical frames.
import numpy as np, nibabel as nib, glob, os
import brkraw as brk

def as_images(conv):
    """Nifti1Images from a convert() result (single image or sequence)."""
    if isinstance(conv, nib.Nifti1Image):
        return [conv]
    try:
        items = list(conv)
    except TypeError:
        items = []
    return [x for x in items if isinstance(x, nib.Nifti1Image)]

def orientation_audit(study, scan, reco=1):
    method = parse_jcamp(f"{study}/{scan}/method")
    acqp   = parse_jcamp(f"{study}/{scan}/acqp")
    visu   = parse_jcamp(f"{study}/{scan}/pdata/{reco}/visu_pars")
    if method is None:
        print(f"  !! no method file for scan {scan}")
        return None
    rec = {"slice_orient": first(val(method, 'PVM_SPackArrSliceOrient')),
           "read_orient":  first(val(method, 'PVM_SPackArrReadOrient')),
           "subj_type":    first(val(visu, 'VisuSubjectType')) if visu else None,
           "subj_pos":     first(val(visu, 'VisuSubjectPosition')) if visu else None,
           "patient_pos":  first(val(acqp, 'ACQ_patient_pos')) if acqp else None,
           "core_dim":     first(val(visu, 'VisuCoreDim')) if visu else None}
    ori = as_list(val(visu, 'VisuCoreOrientation')) if visu else []
    rec["cosines"] = ([round(float(v), 4) for v in ori[:9]] if len(ori) >= 9 else None)
    grad = as_list(val(acqp, 'ACQ_grad_matrix')) if acqp else []
    rec["grad_matrix"] = ([round(float(v), 4) for v in grad[:9]] if len(grad) >= 9 else None)
    imgs = as_images(brk.load(study).convert(scan, reco))
    rec["axcodes_raw"] = nib.aff2axcodes(imgs[0].affine) if imgs else None
    return rec

def audit_all(studies, scans_by_prefix):
    """scans_by_prefix: {'NHP1': 9, 'NHP2': 3}"""
    out = {}
    for idx, study in enumerate(studies):
        pfx = study_prefix(study, idx)
        if pfx not in scans_by_prefix:
            continue
        scan = scans_by_prefix[pfx]
        out[pfx] = orientation_audit(study, scan, 1)
    fields = ["slice_orient", "read_orient", "subj_type", "subj_pos", "patient_pos",
              "core_dim", "axcodes_raw", "grad_matrix", "cosines"]
    print(f"{'field':<14}" + "".join(f"{p:<34}" for p in out))
    print("-" * (14 + 34 * len(out)))
    differs = []
    for f in fields:
        vals = [out[p][f] if out[p] else None for p in out]
        same = all(str(v) == str(vals[0]) for v in vals)
        if not same:
            differs.append(f)
        print(f"{f:<14}" + "".join(f"{str(v)[:32]:<34}" for v in vals) +
              ("" if same else "   <-- DIFFERS"))
    print()
    if not differs:
        print("  all orientation parameters agree. If the NIfTI axcodes still differ, the")
        print("  divergence was introduced by brkraw -> report it.")
    else:
        print(f"  scans differ on: {differs}")
        print("  The parameter files, not brkraw, are the source. The conversion must")
        print("  normalise them to one anatomical frame (ORIENT_TARGET below).")
    return out

# ---- frame correction ---------------------------------------------------------------
ORIENT_TARGET = ("L", "S", "A")     # the anatomical frame every scan must end up in;
                                    # codes are the direction each array axis POINTS TOWARD
ORIENT_OVERRIDE = {                 # {prefix: axcodes the file's affine CURRENTLY claims}
    "NHP2": ("L", "P", "S"),      # set from the audit, confirm visually, then uncomment
}

def frame_correction(current_codes, target_codes):
    """Rigid change of anatomical frame: relabels each array axis from the direction the
    affine claims to the direction it actually points. LEFT-multiplies the affine, so the
    voxel spacing, in-plane direction cosines and obliquity brkraw computed are preserved
    and only the frame they are expressed in changes. Rebuilding the affine from scratch
    would discard all of that."""
    ras = {"R": (0, +1), "L": (0, -1), "A": (1, +1),
           "P": (1, -1), "S": (2, +1), "I": (2, -1)}
    C = np.zeros((3, 3))
    for cur, tgt in zip(current_codes, target_codes):
        r0, s0 = ras[cur]; r1, s1 = ras[tgt]
        C[r1, r0] = s0 * s1
    det = float(np.linalg.det(C))
    if abs(det - 1.0) > 1e-9:
        raise ValueError(
            f"{tuple(current_codes)} -> {tuple(target_codes)} has det={det:+.1f}. "
            f"det=-1 is a MIRROR: it would silently swap left and right. Only proper "
            f"rotations (det=+1) are permitted.")
    M = np.eye(4); M[:3, :3] = C
    return M

def apply_matrix(img, M):
    """A copy of img with M left-multiplied into its affine. Voxel data is not touched."""
    A = M @ img.affine
    out = nib.Nifti1Image(np.asanyarray(img.dataobj), A, img.header)
    out.set_sform(A, code=1); out.set_qform(A, code=1)
    return out

def study_matrix(prefix, target=ORIENT_TARGET):
    """One correction matrix per study, or None. Applies to every scan of that study,
    whatever plane that scan was acquired in."""
    if prefix not in ORIENT_OVERRIDE:
        return None
    return frame_correction(ORIENT_OVERRIDE[prefix], target)

def apply_frame(img, current_codes, target_codes=ORIENT_TARGET):
    """A copy of img in the target anatomical frame. Voxel data is not touched."""
    return apply_matrix(img, frame_correction(current_codes, target_codes))

_AUDIT = audit_all(studies, {"NHP1": 9, "NHP2": 3})

## Convert

Both studies in one pass. The old version required hand-editing `studies[0]` /
`studies[1]` and `OUT` between runs, and hard-coded an `NHP1_` filename prefix for
every study -- which is why a rename cell existed to repair NHP2's names
afterwards. The prefix is now derived from the study folder, so that step is gone.

Each saved file is checked against ground truth as it is written, and header
zooms are re-synced from the affine if they ever disagree.

In [ ]:
# 4. Convert every scan/reco in every study  (patched brkraw)
import os, re, glob
import numpy as np, nibabel as nib
import brkraw as brk

ROOT = '/content/nifti_out'
os.makedirs(ROOT, exist_ok=True)

records = []
for idx, study in enumerate(studies):
    prefix = study_prefix(study, idx)
    _M     = study_matrix(prefix)          # one rotation for the whole study, or None
    out    = os.path.join(ROOT, prefix)
    os.makedirs(out, exist_ok=True)
    raw    = brk.load(study)

    print("=" * 88)
    print(f"{prefix}   {study}")
    print("=" * 88)
    print(f"{'scan':>4} {'reco':>5} {'part':>5} {'shape':>18} {'zooms (mm)':>24} "
          f"{'exp z':>7} {'z':>8}")
    print("-" * 88)

    for sid in raw.avail:
        scan = raw.get_scan(sid)
        exp, _src, _is2d = expected_spacing(study, sid)
        for rid in list(scan.avail):
            try:
                imgs = as_images(raw.convert(sid, rid))
                if not imgs:
                    print(f"{sid:>4} {rid:>5} {'-':>5} <no Nifti1Image returned>")
                    continue
                for i, img in enumerate(imgs):
                    # frame correction: one rigid rotation per study, applied to every scan
                    # regardless of its own plane, BEFORE the zooms are re-synced.
                    if _M is not None:
                        img = apply_matrix(img, _M)
                    # keep header zooms consistent with the affine
                    zaff  = np.linalg.norm(img.affine[:3, :3], axis=0)
                    zooms = list(img.header.get_zooms())
                    if len(zooms) >= 3 and not np.allclose(zooms[:3], zaff, atol=1e-4):
                        zooms[:3] = [float(z) for z in zaff]
                        img.header.set_zooms(tuple(zooms))

                    suffix = f"_part{i}" if len(imgs) > 1 else ""
                    fname  = f"{out}/{prefix}_scan{sid}_reco{rid}{suffix}.nii.gz"
                    nib.save(img, fname)

                    pack   = i if len(imgs) > 1 else 0
                    e      = exp[pack] if pack < len(exp) else (exp[0] if exp else None)
                    zval   = float(zaff[2])
                    mark   = "" if e is None else (" ok" if abs(zval - e) < 1e-3 else " !!")
                    records.append(dict(study=study, prefix=prefix, scan=sid, reco=rid,
                                        part=pack, path=fname, z=zval, expected=e,
                                        axcodes=nib.aff2axcodes(img.affine),
                                        frame_corrected=prefix in ORIENT_OVERRIDE))
                    print(f"{sid:>4} {rid:>5} {(pack if len(imgs) > 1 else '-'):>5} "
                          f"{str(img.shape):>18} "
                          f"{str(tuple(round(float(z), 4) for z in img.header.get_zooms())):>24} "
                          f"{('n/a' if e is None else f'{e:.4f}'):>7} {zval:>7.4f}{mark}")
            except Exception as ex:
                print(f"{sid:>4} {rid:>5} {'-':>5} <{type(ex).__name__}: {ex}>")
    print()

print(f"Saved {len(records)} NIfTI files under {ROOT}")


## Verify

Re-reads every file from disk and compares the affine's z column *and* the header
`pixdim` against ground truth. This reads the saved files rather than the
in-memory objects, so it would catch a bad affine, a header/affine mismatch, or a
silent save problem. Anything other than `PASS` on every row means do not use the
output.

In [ ]:
# 5. Verify saved files against ground truth
import glob, os, re
import numpy as np, nibabel as nib

rows, n_pass, n_fail = [], 0, 0
for f in sorted(glob.glob('/content/nifti_out/**/*.nii*', recursive=True)):
    m = re.search(r'(NHP\d+|study\d+)_scan(\d+)_reco(\d+)(?:_part(\d+))?', os.path.basename(f))
    if not m:
        continue
    prefix, sid, rid = m.group(1), int(m.group(2)), int(m.group(3))
    pack = int(m.group(4)) if m.group(4) else 0

    study = next((s for i, s in enumerate(studies) if study_prefix(s, i) == prefix), None)
    if study is None:
        continue
    exp, src, _ = expected_spacing(study, sid, rid)
    e = exp[pack] if pack < len(exp) else (exp[0] if exp else None)

    img   = nib.load(f)
    z_aff = float(np.linalg.norm(img.affine[:3, 2]))
    z_hdr = float(img.header.get_zooms()[2])
    codes = nib.aff2axcodes(img.affine)
    det   = float(np.linalg.det(img.affine[:3, :3]))
    multi = img.ndim >= 3 and img.shape[2] > 1     # localizer packs are single-slice
    plane = "".join(codes)

    if e is None:
        status = "SKIP (no ground truth)"
    elif abs(z_aff - e) >= 1e-3 or abs(z_hdr - e) >= 1e-3:
        status = "FAIL spacing"; n_fail += 1
    elif det <= 0:
        status = "FAIL left-right mirrored"; n_fail += 1
    elif multi and tuple(codes) != tuple(ORIENT_TARGET):
        status = f"FAIL frame {codes}"; n_fail += 1
    elif not multi:
        status = f"PASS (localizer plane {plane})"; n_pass += 1
    else:
        status = "PASS"; n_pass += 1
    rows.append((os.path.relpath(f, '/content/nifti_out'), e, z_aff, z_hdr, status, src))

print(f"{'file':<44} {'expect':>7} {'affine':>7} {'header':>7}  status")
print("-" * 84)
for name, e, za, zh, st, src in rows:
    print(f"{name:<44} {('n/a' if e is None else f'{e:.4f}'):>7} "
          f"{za:>7.4f} {zh:>7.4f}  {st}")

print("-" * 84)
print(f"{n_pass} passed, {n_fail} failed, {len(rows) - n_pass - n_fail} skipped")
if n_fail:
    print("\n!! FAILURES ABOVE -- do not use this output. Check that the FIX cell "
          "ran before the conversion cell.")
else:
    print("\nAll through-plane spacings match the scanner's own slice positions.")


In [ ]:
# 5b. Apply the same frame correction to already-derived files (DELETE FROM FINAL VERSION)
# =====================================================================================
# The correction changes the affine only, so a mask or crop drawn in a file's own voxel
# space stays valid under it: the voxels do not move, only the frame they are reported in.
# Existing skull-strips, masks and crops therefore do NOT need redrawing -- they need the
# identical matrix applied. Files are written alongside the originals with _frm.
import glob, os, numpy as np, nibabel as nib

DERIVED_TO_FIX = {
    "NHP2": [
         "/content/drive/My Drive/macaque_atlas/scans/nifti_out/NHP2/NHP2_scan3_reco1_skullstripped_cropped.nii.gz",
         "/content/drive/My Drive/macaque_atlas/scans/nifti_out/NHP2/NHP2BrainMask-final.nii"
    ]
}

def fix_derived(mapping=DERIVED_TO_FIX, target=ORIENT_TARGET):
    for prefix, paths in mapping.items():
        M = study_matrix(prefix, target)
        if M is None:
            print(f"  {prefix}: no ORIENT_OVERRIDE entry -- nothing to correct"); continue
        for p in paths:
            if not os.path.exists(p):
                print(f"  !! missing: {p}"); continue
            img  = nib.load(p)
            was  = nib.aff2axcodes(img.affine)
            out  = apply_matrix(img, M)
            base = p[:-7] if p.endswith(".nii.gz") else os.path.splitext(p)[0]
            dst  = f"{base}_frm.nii.gz"
            nib.save(out, dst)
            now  = nib.aff2axcodes(out.affine)
            print(f"  {os.path.basename(p)}: {was} -> {now}  ({os.path.basename(dst)})")

fix_derived()

In [ ]:
# 6. Package and download
import shutil, os
from google.colab import files

shutil.make_archive('/content/nifti_out', 'zip', '/content/nifti_out')
print("zip size:", os.path.getsize('/content/nifti_out.zip'), "bytes")
files.download('/content/nifti_out.zip')
